# Module 19 — Evaluation That Actually Means Something

A CPU-only, $0 tour of the evaluation **failure modes**, made visceral with numbers and plots.
Every cell uses the module's pure functions (`metrics`, `contamination`, `judge`, `benchmarks`) —
no GPU, no network, no model download.

The lesson of the module in one sentence: **a benchmark number is an estimate from an instrument
with a bias and an error bar, and evaluation is the discipline of knowing both.**

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import metrics as M
import contamination as C
import judge as J
import benchmarks as B

rng = np.random.default_rng(0)
print("modules loaded — pure Python, no torch needed")

modules loaded — pure Python, no torch needed


## 1. The error bar nobody reports

"62%" is a *sample mean*. With only a few hundred problems the 95% confidence interval is wide —
wide enough that most 1–2 point leaderboard gaps are noise.

In [2]:
# A 200-problem eval that scored 62%.
n = 200
scores = (rng.random(n) < 0.62).astype(float)
ci = M.bootstrap_ci(scores, seed=0)
wil = M.wilson_interval(int(scores.sum()), n)
print(f"bootstrap : {ci}")
print(f"wilson    : {wil}")
print(f"half-width: +/-{ci.half_width*100:.1f} points on a {n}-item eval")
print()
# How many problems would you need for a tight CI?
for hw in (0.05, 0.03, 0.02, 0.01):
    print(f"  +/-{hw*100:>4.1f}pt CI at p=0.62 needs {M.min_n_for_halfwidth(0.62, hw):>6,} problems")

bootstrap : 0.565 [0.495, 0.635] (95% CI)
wilson    : 0.565 [0.496, 0.632] (95% CI)
half-width: +/-7.0 points on a 200-item eval

  +/- 5.0pt CI at p=0.62 needs    363 problems
  +/- 3.0pt CI at p=0.62 needs  1,006 problems
  +/- 2.0pt CI at p=0.62 needs  2,263 problems
  +/- 1.0pt CI at p=0.62 needs  9,051 problems


In [3]:
# Visualize: CI half-width shrinks as 1/sqrt(n). Tight bars are expensive.
ns = np.array([50, 100, 200, 500, 1000, 2000, 5000])
halfs = []
for nn in ns:
    s = (rng.random(nn) < 0.62).astype(float)
    c = M.bootstrap_ci(s, n_resamples=2000, seed=1)
    halfs.append(c.half_width * 100)
plt.figure(figsize=(7,4))
plt.plot(ns, halfs, "o-")
plt.axhline(2.0, ls="--", c="r", label="+/-2pt target")
plt.xscale("log"); plt.xlabel("number of problems"); plt.ylabel("95% CI half-width (points)")
plt.title("Error bar vs eval size — small benchmarks can't resolve small gaps")
plt.legend(); plt.tight_layout(); plt.show()

/tmp/claude-1000/ipykernel_3038/1574296776.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.tight_layout(); plt.show()


## 2. Comparing two models: pair, or fool yourself

Both models saw the *same* problems, so the per-item scores are **paired**. The paired difference
CI is far tighter than comparing two independent marginal CIs, because problem difficulty cancels.

In [4]:
# B = A with a few problems fixed. A genuine but small improvement.
a = (rng.random(300) < 0.60).astype(float)
b = a.copy()
lost = np.where(a == 0)[0]
b[lost[:18]] = 1.0          # B fixes 18 problems A got wrong
won = np.where(a == 1)[0]
b[won[:4]] = 0.0            # B breaks 4 A got right

cmp = M.paired_bootstrap_diff(a, b, seed=0)
ci_a, ci_b = M.bootstrap_ci(a, seed=0), M.bootstrap_ci(b, seed=0)
print(f"A = {cmp.mean_a:.1%}   marginal CI [{ci_a.lo:.1%}, {ci_a.hi:.1%}]")
print(f"B = {cmp.mean_b:.1%}   marginal CI [{ci_b.lo:.1%}, {ci_b.hi:.1%}]   (marginals OVERLAP)")
print(f"PAIRED diff = {cmp.diff:+.1%}  CI [{cmp.diff_ci.lo:+.1%}, {cmp.diff_ci.hi:+.1%}]")
print(f"McNemar p   = {cmp.p_value:.4f}   -> {'SIGNIFICANT' if cmp.significant else 'not significant'}")
print(f"(B better on {cmp.n_b_better}, A better on {cmp.n_a_better}, tie {cmp.n_tie})")

A = 56.7%   marginal CI [51.0%, 62.3%]
B = 61.3%   marginal CI [55.7%, 66.7%]   (marginals OVERLAP)
PAIRED diff = +4.7%  CI [+1.7%, +7.7%]
McNemar p   = 0.0043   -> SIGNIFICANT
(B better on 18, A better on 4, tie 278)


In [5]:
# The marginal CIs overlap (looks inconclusive); the paired CI excludes 0 (it's real).
fig, ax = plt.subplots(figsize=(7,3.2))
ax.errorbar([ci_a.point], [2], xerr=[[ci_a.point-ci_a.lo],[ci_a.hi-ci_a.point]], fmt="o", label="A (marginal)", capsize=5)
ax.errorbar([ci_b.point], [2.3], xerr=[[ci_b.point-ci_b.lo],[ci_b.hi-ci_b.point]], fmt="o", label="B (marginal)", capsize=5)
ax.errorbar([0.5+cmp.diff], [1], xerr=[[cmp.diff-cmp.diff_ci.lo],[cmp.diff_ci.hi-cmp.diff]], fmt="s", c="green", label="paired diff (recentred)", capsize=5)
ax.axvline(0.5, ls="--", c="gray")
ax.set_yticks([1,2,2.3]); ax.set_yticklabels(["paired B-A","A","B"])
ax.set_xlabel("accuracy (and recentred diff)"); ax.set_title("Paired beats unpaired: the diff CI is what decides")
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

/tmp/claude-1000/ipykernel_3038/512554642.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(loc="lower right"); plt.tight_layout(); plt.show()


## 3. Sampling metrics: pass@k / avg@k / maj@k

When the model samples, one decode is one draw. The unbiased pass@k estimator lets you draw `n`
samples once and read off pass@1, pass@5, pass@k from the same draws.

In [6]:
ks = list(range(1, 21))
plt.figure(figsize=(7,4))
for c in (1, 2, 4):
    ys = [M.pass_at_k(20, c, k) for k in ks]
    plt.plot(ks, ys, "o-", label=f"c={c}/20 correct")
plt.xlabel("k (samples drawn)"); plt.ylabel("pass@k"); plt.ylim(0,1.05)
plt.title("pass@k: more attempts, more chances — unbiased estimator (Codex)")
plt.legend(); plt.tight_layout(); plt.show()

# maj@k vs avg@k on a noisy reasoner: correct answer is the modal attractor.
samples = [42, 42, 7, 42, 13, 42, 7]        # 4x correct(42), errors diffuse
print(f"avg@k  = {M.avg_at_k([1.0 if s==42 else 0.0 for s in samples]):.2f}  (fraction correct)")
print(f"maj@k  = {M.majority_at_k(samples, 42)}  (modal answer correct? -> beats avg@k)")

avg@k  = 0.57  (fraction correct)
maj@k  = True  (modal answer correct? -> beats avg@k)


/tmp/claude-1000/ipykernel_3038/1909385665.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.tight_layout(); plt.show()


## 4. Multiple choice: the score moves when the *harness* moves

Same model, same data. Change the **normalization** (raw sum / per-token / per-byte) or the
**prompt format** (lettered vs cloze) and the accuracy changes by several points.

In [7]:
# One question, mock per-option logprobs. Raw favors the SHORT option; per-token/byte
# can favor a longer one. (model.continuation_logprob produces these on a real model.)
logprobs = [-2.0, -3.6, -5.0, -4.2]
toks     = [1,    3,    5,    4]
byts     = [3,    15,   24,   19]
for norm in ("raw", "token", "byte"):
    print(f"  argmax under {norm:5s} -> option {B.score_mc(logprobs, toks, byts, norm)}")

  argmax under raw   -> option 0
  argmax under token -> option 2
  argmax under byte  -> option 2


In [8]:
# Accuracy on a synthetic 200-item, 4-option set where correct answers tend to be LONGER.
# A model with a mild per-token preference for the right answer. Raw normalization penalizes
# the (longer) correct options; per-token recovers them.
NI, NO = 200, 4
gold = rng.integers(0, NO, NI)
accs = {"raw": 0, "token": 0, "byte": 0}
preds = {k: [] for k in accs}
exs = []
for i in range(NI):
    tcounts = rng.integers(1, 6, NO)
    tcounts[gold[i]] += rng.integers(2, 5)            # correct option is longer
    per_tok = rng.normal(-2.0, 0.3, NO)
    per_tok[gold[i]] += 0.35                           # mild true signal per token
    lp = per_tok * tcounts
    bcounts = tcounts * 4
    exs.append(B.MCExample("q", ["o"]*NO, int(gold[i])))
    for norm in accs:
        preds[norm].append(B.score_mc(list(lp), list(tcounts), list(bcounts), norm))
res = {norm: float(np.mean(B.mc_accuracy(preds[norm], exs))) for norm in accs}
print("accuracy by normalization:", {k: round(v,3) for k,v in res.items()})

plt.figure(figsize=(6,4))
plt.bar(list(res), [v*100 for v in res.values()], color=["#c44","#4a4","#46c"])
plt.axhline(25, ls="--", c="gray", label="chance (4-way)")
plt.ylabel("accuracy (%)"); plt.title("Same model & data — normalization swings the score")
plt.legend(); plt.tight_layout(); plt.show()

accuracy by normalization: {'raw': 0.035, 'token': 0.58, 'byte': 0.58}


/tmp/claude-1000/ipykernel_3038/4019627064.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.tight_layout(); plt.show()


## 5. LLM-as-judge: position bias, length bias, and calibration

The judge is the workhorse of post-training eval and the easiest to fool. We use the deterministic
`DummyJudge` so the biases are exactly reproducible.

In [9]:
# A judge that always picks the FIRST-shown answer = pure position bias.
# Running both orders (swap) catches it and scores those pairs as ties.
class AlwaysFirst:
    def __call__(self, prompt): return '{"winner": "A"}'
af = AlwaysFirst()
v_noswap = [J.pairwise_judge(af, "q", "ans one", "ans two", swap=False) for _ in range(20)]
v_swap   = [J.pairwise_judge(af, "q", "ans one", "ans two", swap=True)  for _ in range(20)]
r_noswap = J.aggregate_win_rate(v_noswap)
r_swap   = J.aggregate_win_rate(v_swap)
print(f"no-swap:  A appears to win everything -> win_rate(B)={r_noswap.win_rate:.2f}, pos_bias_detected={r_noswap.position_bias_rate:.0%}")
print(f"swap:     bias caught -> win_rate(B)={r_swap.win_rate:.2f} (ties), pos_bias_detected={r_swap.position_bias_rate:.0%}")

no-swap:  A appears to win everything -> win_rate(B)=0.00, pos_bias_detected=0%
swap:     bias caught -> win_rate(B)=0.50 (ties), pos_bias_detected=100%


In [10]:
# Length bias: a judge that prefers the longer answer. Mix which side is longer so the
# win pattern correlates with length. The correlation is the alarm.
lj = J.DummyJudge(rule="longer")
pairs = [("q","short","a considerably longer and more verbose response here"),
         ("q","another quite long and elaborate reply that goes on","tiny"),
         ("q","no","yes indeed absolutely certainly definitely so"),
         ("q","the most detailed and complete explanation possible","yep")]
vs = [J.pairwise_judge(lj, q, a, b) for q,a,b in pairs]
la = [len(a) for _,a,_ in pairs]; lb = [len(b) for _,_,b in pairs]
res = J.aggregate_win_rate(vs, la, lb)
print(f"win_rate(B)={res.win_rate:.2f}  length-win corr={res.length_win_corr:+.2f}  (>>0 => length artifact)")
print()
# Calibration: raw agreement lies on skewed labels; Cohen's kappa corrects it.
judge_says = ["B"]*10
humans     = ["B"]*8 + ["A"]*2
ag = J.agreement(judge_says, humans)
print(f"a 'always B' judge on 80/20 data: raw agreement={ag.raw_agreement:.0%}, kappa={ag.cohen_kappa:.2f}, trustworthy={ag.trustworthy}")
good = J.agreement(["A","B","B","A","tie","B","A"], ["A","B","A","A","tie","B","A"])
print(f"a real judge:                     raw agreement={good.raw_agreement:.0%}, kappa={good.cohen_kappa:.2f}, trustworthy={good.trustworthy}")

win_rate(B)=0.50  length-win corr=+1.00  (>>0 => length artifact)

a 'always B' judge on 80/20 data: raw agreement=80%, kappa=0.00, trustworthy=False
a real judge:                     raw agreement=86%, kappa=0.77, trustworthy=True


## 6. Contamination: when the test set is in the training set

N-gram overlap flags verbatim leakage; a canary GUID flags ingested benchmark files. The only
real defense is a private, freshly-authored eval set.

In [11]:
# A training corpus that happens to contain one benchmark question verbatim + a canary.
corpus = [
    "the mitochondria is the powerhouse of the cell and produces atp for energy",
    "janet has sixteen eggs per day and sells the remainder at the market for two dollars",
    f"internal benchmark dump {C.BIG_BENCH_CANARY} please do not train on this file",
] + [f"unrelated web document number {i} about various everyday topics" for i in range(30)]

test_items = [
    "Janet has sixteen eggs per day and sells the remainder at the market for two dollars",  # LEAKED
    "A spacecraft accelerates toward Mars using an ion drive over many months",              # clean
    "The mitochondria is the powerhouse of the cell",                                         # partial
    "Photosynthesis converts sunlight into chemical energy in plant chloroplasts",            # clean
]
rep = C.contamination_report(test_items, corpus, n=6, threshold=0.5)
print(rep.summary())
for i, ov in enumerate(rep.overlaps):
    print(f"  test[{i}] overlap={ov:.2f}  flagged={i in rep.flagged_indices}")
canary = C.find_canary(corpus)
print(f"\nCANARY found in corpus docs: {canary}  -> benchmark files leaked into training")
print(f"overfit gap example (public 85% vs private 72%): {C.overfit_gap(0.85, 0.72):.0%}")

contamination: 2/4 items flagged (50.0%) at 6-gram overlap >= 50%; mean overlap 50.0%
  test[0] overlap=1.00  flagged=True
  test[1] overlap=0.00  flagged=False
  test[2] overlap=1.00  flagged=True
  test[3] overlap=0.00  flagged=False

CANARY found in corpus docs: [2]  -> benchmark files leaked into training
overfit gap example (public 85% vs private 72%): 13%


In [12]:
plt.figure(figsize=(6,4))
colors = ["#c44" if i in rep.flagged_indices else "#4a4" for i in range(len(rep.overlaps))]
plt.bar(range(len(rep.overlaps)), [o*100 for o in rep.overlaps], color=colors)
plt.axhline(50, ls="--", c="gray", label="flag threshold (50%)")
plt.xlabel("test item"); plt.ylabel("n-gram overlap with corpus (%)")
plt.title("Contamination scan — red items leaked verbatim into training")
plt.legend(); plt.tight_layout(); plt.show()

/tmp/claude-1000/ipykernel_3038/2217610727.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.tight_layout(); plt.show()


## Recap

| Failure mode | The fix in this module |
|---|---|
| Score with no error bar | `metrics.bootstrap_ci` / `wilson_interval` — always report a CI |
| "B beats A" from overlapping marginals | `metrics.paired_bootstrap_diff` + `mcnemar_test` — pair the comparison |
| Single number on a tiny set (AIME) | `pass_at_k` / `avg_at_k` / `majority_at_k` |
| MMLU differs across papers | pin normalization + format (`benchmarks.score_mc`, `build_mc_prompt`) |
| Judge measures position/length | `pairwise_judge(swap=True)` + length-win corr |
| Trusting an uncalibrated judge | `judge.agreement` — Cohen's kappa vs humans |
| High score that doesn't generalize | `contamination` scan + a private held-out set |

**Public benchmarks orient you. Private evals, paired tests, calibrated judges, and contamination
scans are what you ship on.** Run `eval.py --config=configs/eval_qwen3_1.7b.yaml` to produce a real
scorecard — with every number carrying its confidence interval.